## Michaud Model

$\frac{\partial RT}{\partial t}=R(RT,RD,F)+D_{RT}\triangle RT$

$\frac{\partial RD}{\partial t}=k_5-k_6RD-R(RT,RD,F)+D_{RD}\triangle RD$

$\frac{\partial F}{\partial t}=k_7+k_8\frac{RT^2}{1+k_9RT^2}-k_{10}dW(\sigma,s)F+D_F\triangle F$

where $R$ is the reaction function:

$R(*)=(k_0+\alpha\frac{k_1RT^3}{1+k_2RT^2})RD-(k_3+k_4(1+\beta)F)RT$

In [5]:
#This code does the same thing as the michaud code but saves it as a 3D numpy array instead

import numpy as np
from pde_utils import correlated_gaussian_field, laplacian

# set random seed
np.random.seed(42)

# define parameters
k0 = 0.00625
k1 = 0.3125
k2 = 1
k3 = 0.0625
k4 = 0.05625
k5 = 0.0625
k6 = 0.02083
k7 = 0.001875
k8 = 0.14062*2
k9 = 0.25
k10 = 0.025
Drt = 0.08
Drd = 0.4
Df = 0.8
sigma = 0.75
s = 4 
f = 10 # dW update frequency
alpha = 1
beta = 1

size = 100    # number of cells
dt = 0.1     # time step
t_total = 1000.0  # run time
frame_int = 25    # seconds between saved frames

# initial fields
RT = 0.1 + 0.9 * np.random.rand(size, size)
RD = np.full((size, size), 0.1)
F  = np.zeros((size, size))

# initial noise
dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

def reaction(A, B, C):
    return (k0 + alpha*k1*A**3/(1 + k2*A**2))*B - (k3 + k4*(1+beta)*C)*A

# list to collect 2D frames
frames = []

n_steps      = int(t_total / dt)
save_every   = 5
dW_update    = int(f / dt)

for i in range(n_steps):
    R = reaction(RT, RD, F)

    RT = RT + dt * (R     + Drt * laplacian(RT))
    RD = RD + dt * (k5 - k6*RD - R + Drd * laplacian(RD))
    F  = F  + dt * (k7 + k8*RT**2/(1 + k9*RT**2) - k10*dW*F + Df * laplacian(F))

    # refresh noise
    if i % dW_update == 0:
        dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

    # save a frame
    if i % save_every == 0:
        frames.append(RT.copy())

# stack into a single 3D array: (n_frames, size, size)
frames_array = np.stack(frames, axis=0)

# Save
np.save('michaud_simulation_saved_standing.npy', frames_array)

print(f"Saved {frames_array.shape[0]} frames→ 'michaud_simulation.npy'; array shape = {frames_array.shape}")


Saved 2000 frames→ 'michaud_simulation.npy'; array shape = (2000, 100, 100)
